# Language mix — how multilingual is the stream?

Using the `language` column we can watch the balance of languages evolve
and see whether certain languages arrive in bursts.

*SQL is kept inline in this notebook.*

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
import psycopg
from dotenv import load_dotenv

# Connection settings come from the repository root .env
load_dotenv(Path("..") / ".env")

conninfo = psycopg.conninfo.make_conninfo(
    host=os.environ["POSTGRES_HOST"],
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname=os.environ["POSTGRES_DATABASE"],
    sslmode=os.getenv("PGSSLMODE", "require"),
)


def q(sql: str, **params) -> pd.DataFrame:
    """Run a query and return a DataFrame. Params are passed safely to psycopg."""
    with psycopg.connect(conninfo) as conn, conn.cursor() as cur:
        cur.execute(sql, params or None)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


print("Ready — connected settings for", os.environ["POSTGRES_HOST"])


## Totals per language

In [ ]:
totals = q("""
    SELECT language, COUNT(*) AS n
    FROM ngrams
    GROUP BY language
    ORDER BY n DESC
""")

fig = px.bar(totals, x="language", y="n", color="language",
             title="Total occurrences per language")
fig.update_layout(height=400, showlegend=False, yaxis_title="occurrences")
fig.show()

## Composition over time (100% stacked)
Each day normalised to 100% — this shows *share*, not absolute volume, so
shifts in the language mix pop out even on quiet days.

In [ ]:
daily = q("""
    SELECT date_trunc('day', timestamp) AS day,
           language,
           COUNT(*)                     AS n
    FROM ngrams
    GROUP BY 1, 2
    ORDER BY 1
""")

fig = px.area(
    daily, x="day", y="n", color="language",
    groupnorm="fraction",
    title="Language composition over time (share of daily volume)",
)
fig.update_layout(height=460, xaxis_title="", yaxis_title="share",
                  yaxis_tickformat=".0%")
fig.show()